# E01: Comprehensions y Generadores - Código Pythonic

## Objetivos de Aprendizaje

Al finalizar este notebook, serás capaz de:

1. **Dominar** las comprehensions unidimensionales (list, dict, set) con y sin filtros, incluyendo el operador walrus `:=`.
2. **Diseñar** comprehensions anidadas para transformar datos tabulares y producto cartesiano, controlando el orden de evaluación.
3. **Comprender** la diferencia fundamental entre expresiones generadoras (lazy) y listas (eager), y cuándo usar cada una.
4. **Implementar** funciones generadoras con `yield`, `yield from` y estado persistente.
5. **Construir** generadores infinitos seguros y combinarlos con `itertools` para pipelines eficientes.

## Analogía: Plantillas de Troquelado y Grifos Inteligentes

Imagina una **fábrica de galletas**:

- **Comprehensions** son como **plantillas de troquelado**: defines una forma (la expresión) y una masa de ingredientes (la secuencia). En un solo paso obtienes **todas** las galletas listas en la bandeja. Es rápido, pero si necesitas 1 millón de galletas, ocuparás mucho espacio en la cocina (memoria).

- **Generadores** son como un **grifo inteligente** que produce una gota a vez. No llenas un vaso, solo abres el grifo cuando necesitas una gota. Puedes tener un "grifo infinito" (como el de los ejemplos) y nunca se agota, porque solo produce lo que le pides en cada momento.

```
Comprehension (Eager)     Generador (Lazy)
┌─────────────────┐       ┌─────────────────┐
│ 🔵🔵🔵🔵🔵🔵🔵 │       │ 🔵 → 🔵 → 🔵 → 🔵
│ 🔵🔵🔵🔵🔵🔵🔵 │       │                 │
│ 🔵🔵🔵🔵🔵🔵🔵 │       │ Producción bajo │
│ Producción total │       │ demanda (gotas) │
│ memoria alta     │       │ memoria mínima  │
└─────────────────┘       └─────────────────┘
```

## 1. Comprehensions Unidimensionales

Las comprehensions son azúcar sintáctico para construir secuencias de forma declarativa. La forma general es:

```python
[expresión for ítem in iterable if condición]
```

Veamos cada tipo.

### 1.1 List Comprehension

In [ ]:
# Tradicional vs comprehension
numeros = range(1, 11)

# Forma tradicional
cuadrados_tradicional = []
for n in numeros:
    cuadrados_tradicional.append(n ** 2)

# Forma pythonic
cuadrados = [n ** 2 for n in numeros]

print(f"Tradicional: {cuadrados_tradicional}")
print(f"Comprehension: {cuadrados}")

In [ ]:
# Con condición: solo pares
pares = [n for n in range(1, 21) if n % 2 == 0]
print(f"Pares del 1 al 20: {pares}")

# Transformación + filtro
raices_enteras = [round(n ** 0.5, 2) for n in range(1, 11) if n % 3 == 0]
print(f"Raíces de múltiplos de 3: {raices_enteras}")

### 1.2 Operador Walrus `:=` en Comprehensions

Desde Python 3.8, el walrus operator (`:=`) permite **asignar y usar** una variable en la misma expresión. Esto evita recalcular valores costosos.

In [ ]:
import math

datos = [2, -15, 3.7, 0, 8.9, -4.2, 16, 100]

# Sin walrus: calculamos raíz dos veces
resultados_v1 = [math.sqrt(abs(x)) for x in datos if math.sqrt(abs(x)) > 3]

# Con walrus: calculamos una vez, reutilizamos
resultados_v2 = [
    r
    for x in datos
    if (r := math.sqrt(abs(x))) > 3  # asigna y filtra
]

print(f"Sin walrus (recalcula): {resultados_v1}")
print(f"Con walrus (eficiente): {resultados_v2}")

In [ ]:
# Walrus para procesar texto: extraer longitudes > 5
palabras = ["sol", "programación", "sol", "datos", "python", "sol", "machine learning"]

largas = [
    f"{palabra}({l})"
    for palabra in palabras
    if (l := len(palabra)) > 5
]

print(f"Palabras largas con longitud: {largas}")

### 1.3 Dict Comprehension

In [ ]:
# Cuadrados como diccionario
cuad_dict = {n: n**2 for n in range(1, 11)}
print(f"Cuadrados: {cuad_dict}")

# Invertir diccionario
inverted = {v: k for k, v in cuad_dict.items()}
print(f"Invertido: {inverted}")

# Filtrar por condición del valor
pares_dict = {k: v for k, v in cuad_dict.items() if v % 2 == 0}
print(f"Solo pares: {pares_dict}")

In [ ]:
# Crear diccionario desde dos listas
nombres = ["Ana", "Luis", "María", "Pedro"]
notas = [9.5, 8.0, 9.8, 7.5]

boletin = {nombre: nota for nombre, nota in zip(nombres, notas)}
print(f"Boletín: {boletin}")

# Clasificar por condición
aprobados = {k: v for k, v in boletin.items() if v >= 8.0}
print(f"Aprobados: {aprobados}")

### 1.4 Set Comprehension

In [ ]:
# Eliminar duplicados y transformar
textos = ["Python", "python", "PYTHON", "java", "Java", "JAVA"]

lenguajes_unicos = {t.lower() for t in textos}
print(f"Lenguajes únicos (orden arbitrario): {lenguajes_unicos}")

# Longitudes únicas
longitudes = {len(palabra) for palabra in "La programación es divertida".split()}
print(f"Longitudes únicas: {longitudes}")

---

## 2. Comprehensions Anidadas

El **orden de los bucles** en una comprehension anidada es idéntico al de los bucles `for` tradicionales: **de izquierda a derecha, el más externo primero**.

### 2.1 Matrices y Transposición

In [ ]:
# Crear matriz 3x3
matriz = [[i * 3 + j + 1 for j in range(3)] for i in range(3)]
print("Matriz 3x3:")
for fila in matriz:
    print(f"  {fila}")

# Transposición
transpuesta = [[fila[i] for fila in matriz] for i in range(3)]
print("\nTranspuesta:")
for fila in transpuesta:
    print(f"  {fila}")

In [ ]:
# Transposición con zip (más idiomático)
transpuesta_zip = [list(fila) for fila in zip(*matriz)]
print("Transpuesta con zip:")
for fila in transpuesta_zip:
    print(f"  {fila}")

### 2.2 Aplanar una Matriz

In [ ]:
# Aplanar matriz 2D
matriz_2d = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

# El bucle externo itera filas, el interno elementos
aplanada = [elem for fila in matriz_2d for elem in fila]
print(f"Original: {matriz_2d}")
print(f"Aplanada: {aplanada}")

# ¡CUIDADO con el orden! Esto es diferente:
# [fila for elem in matriz_2d for fila in elem]  # ¡Error de lógica!

### 2.3 Producto Cartesiano

In [ ]:
colores = ["rojo", "azul"]
tamaños = ["S", "M", "L"]

# Producto cartesiano
combinaciones = [(c, t) for c in colores for t in tamaños]
print("Producto cartesiano:")
for combo in combinaciones:
    print(f"  {combo}")

print(f"\nTotal combinaciones: {len(combinaciones)}")

In [ ]:
# Ejemplo práctico: generar configuraciones de prueba
entornos = ["dev", "staging", "prod"]
tipos_db = ["postgres", "mysql"]
niveles_log = ["DEBUG", "INFO", "WARNING"]

configs = [
    {"env": e, "db": d, "log": l}
    for e in entornos
    for d in tipos_db
    for l in niveles_log
    if not (e == "prod" and l == "DEBUG")  # filtro inteligente
]

print(f"Configs generadas: {len(configs)}")
for c in configs[:3]:  # primeras 3
    print(f"  {c}")

---

## 3. Expresiones Generadoras

Una expresión generadora tiene la **misma sintaxis** que una list comprehension, pero usa **paréntesis** `()` en lugar de corchetes `[]`. La diferencia es fundamental: **no crea la lista en memoria**, sino que produce elementos bajo demanda.

In [ ]:
import sys

# List comprehension: crea toda la lista en memoria
lista = [x ** 2 for x in range(1_000_000)]
print(f"Tamaño de lista en memoria: {sys.getsizeof(lista):,} bytes")

# Generador expression: solo almacena la fórmula
generador = (x ** 2 for x in range(1_000_000))
print(f"Tamaño del generador: {sys.getsizeof(generador)} bytes")
print(f"\nRatio de compresión: {sys.getsizeof(lista) / sys.getsizeof(generador):,.0f}x más pequeño")

In [ ]:
# Iterar directamente (sin materializar)
suma_gen = sum(x ** 2 for x in range(1_000_000))
print(f"Suma de cuadrados del 0 al 999,999: {suma_gen:,}")

In [ ]:
# next(): obtener el siguiente elemento
gen = (x * 10 for x in range(5))

print(f"Primer elemento: {next(gen)}")
print(f"Segundo elemento: {next(gen)}")
print(f"Tercer elemento: {next(gen)}")
print(f"Cuarto: {next(gen)}, Quinto: {next(gen)}")

# Intentar obtener más lanza StopIteration
try:
    next(gen)
except StopIteration:
    print("\n¡StopIteration! El generador se agotó.")

In [ ]:
# Convertir a lista cuando necesitas reutilizar
gen_temp = (x ** 3 for x in range(10))
lista_final = list(gen_temp)
print(f"Lista materializada: {lista_final}")

# ¡Cuidado! El generador ya se agotó
print(f"Volver a convertir: {list(gen_temp)}  # ¡Lista vacía!")

---

## 4. Funciones Generadoras

Las funciones generadoras usan `yield` en lugar de `return`. Cada vez que Python encuentra un `yield`, **pausa** la ejecución y devuelve el valor. La próxima llamada **retoma** exactamente donde se quedó.

In [ ]:
def countdown(n):
    """Generador de cuenta regresiva con estado persistente."""
    print(f"  → Iniciando cuenta regresiva desde {n}")
    while n > 0:
        yield n  # pausa aquí, devuelve n
        print(f"  → Retomando después de yield, n={n}")
        n -= 1
    print("  → ¡Despegue!")

# El generador NO ejecuta nada todavía
gen = countdown(3)
print("Generador creado, sin ejecutar")
print()

# Cada next() avanza hasta el siguiente yield
print(f"1er next(): {next(gen)}")
print(f"2do next(): {next(gen)}")
print(f"3er next(): {next(gen)}")
try:
    print(f"4to next(): {next(gen)}")
except StopIteration:
    print("StopIteration lanzado")

In [ ]:
# Generador con lógica acumulativa
def acumulador_impares(inicio, fin):
    """Produce impares y lleva cuenta de la suma acumulada."""
    suma = 0
    for n in range(inicio, fin + 1):
        if n % 2 != 0:
            suma += n
            yield n, suma

print("Impares con suma acumulada:")
for valor, acumulado in acumulador_impares(1, 10):
    print(f"  valor={valor}, acumulado={acumulado}")

### 4.1 `yield from`: Delegación de Generadores

In [ ]:
def rango_letras(inicio, fin):
    """Genera letras desde inicio hasta fin."""
    for codigo in range(ord(inicio), ord(fin) + 1):
        yield chr(codigo)

def combinador(*generadores):
    """Delega a múltiples generadores usando yield from."""
    for gen in generadores:
        yield from gen  # delega sin crear lista intermedia

# Combinar rangos de letras
secuencia = combinador(
    rango_letras('a', 'c'),
    rango_letras('x', 'z'),
    rango_letras('A', 'C'),
)

print("Secuencia combinada:")
for letra in secuencia:
    print(f"  {letra}", end="")
print()

In [ ]:
# Aplicación real: generar datos de prueba en capas
def baseusuarios(n):
    for i in range(1, n + 1):
        yield {"id": i, "tipo": "base"}

def usuariospremium(inicio, cantidad):
    for i in range(inicio, inicio + cantidad):
        yield {"id": i, "tipo": "premium"}

def todos_los_usuarios(n_base, m_premium):
    yield from baseusuarios(n_base)
    yield from usuariospremium(n_base + 1, m_premium)

print("Primeros usuarios generados:")
for u in todos_los_usuarios(3, 2):
    print(f"  {u}")

---

## 5. Generadores Infinitos y Perezosos

Los generadores pueden ser **infinitos** porque solo producen un valor a la vez. Nunca debes convertir un generador infinito a lista.

In [ ]:
from itertools import islice

def contar_desde(n=0, paso=1):
    """Generador infinito que cuenta desde n."""
    while True:
        yield n
        n += paso

# ¡NUNCA hacer: list(contar_desde())!
# Usar islice para tomar N elementos
print("Primeros 10 desde 5:")
for valor in islice(contar_desde(5), 10):
    print(f"  {valor}", end="")
print()

In [ ]:
def fibonacci():
    """Generador infinito de Fibonacci."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

# Los primeros 15 números de Fibonacci
fibs = list(islice(fibonacci(), 15))
print(f"Fibonacci: {fibs}")

# Encontrar el primer Fibonacci mayor a 1000
gen_fib = fibonacci()
for fib in gen_fib:
    if fib > 1000:
        print(f"\nPrimer Fibonacci > 1000: {fib}")
        break

In [ ]:
def sieve_of_eratosthenes():
    """Generador infinito de números primos (Criba optimizada)."""
    yield 2
    composiciones = {}
    n = 3
    while True:
        if n not in composiciones:
            yield n
            composiciones[n * n] = [n]
        else:
            for p in composiciones[n]:
                composiciones.setdefault(n + 2 * p, []).append(p)
            del composiciones[n]
        n += 2

# Los primeros 20 primos
primos = list(islice(sieve_of_eratosthenes(), 20))
print(f"Primeros 20 primos: {primos}")

---

## 6. Ahorro de Memoria en la Práctica

La diferencia de memoria es **exponencial** con datos grandes. Veamos la comparación real.

In [ ]:
import sys
import time

def comparar_memoria(tamano):
    """Compara memoria entre lista y generador."""
    # Lista: almacena todo
    lista = [x ** 2 for x in range(tamano)]
    memoria_lista = sys.getsizeof(lista)
    # + el contenido real de los enteros (aproximado)
    memoria_lista_total = memoria_lista + tamano * sys.getsizeof(0)
    
    # Generador: solo la fórmula
    generador = (x ** 2 for x in range(tamano))
    memoria_gen = sys.getsizeof(generador)
    
    return memoria_lista_total, memoria_gen

print(f"{'Elementos':>12} {'Lista (MB)':>12} {'Generador':>12} {'Ratio':>8}")
print("-" * 50)
for n in [1_000, 10_000, 100_000, 1_000_000]:
    mem_list, mem_gen = comparar_memoria(n)
    print(f"{n:>12,} {mem_list / 1024 / 1024:>10.2f}MB {mem_gen:>10}B {mem_list / mem_gen:>7.0f}x")

In [ ]:
# Benchmark: tiempo de creación vs iteración
n = 5_000_000

# Tiempo de CREACIÓN
t0 = time.perf_counter()
lista_grande = [x ** 2 for x in range(n)]
t_lista = time.perf_counter() - t0

t0 = time.perf_counter()
gen_grande = (x ** 2 for x in range(n))
t_gen = time.perf_counter() - t0

print(f"Creación de {n:,} elementos:")
print(f"  Lista: {t_lista:.4f}s")
print(f"  Generador: {t_gen:.10f}s (casi instantáneo)")

# Tiempo de ITERACIÓN completa
t0 = time.perf_counter()
_ = sum(lista_grande)
t_iter_lista = time.perf_counter() - t0

t0 = time.perf_counter()
_ = sum(gen_grande)  # se agota aquí
t_iter_gen = time.perf_counter() - t0

print(f"\nIteración completa:")
print(f"  Lista: {t_iter_lista:.4f}s (ya estaba en memoria)")
print(f"  Generador: {t_iter_gen:.4f}s (calcula bajo demanda)")

In [ ]:
# ¿Cuándo usar cada uno?
print("""
┌─────────────────────────────────────────────────────────────────┐
│              ¿CUÁNDO USAR CADA ENFOQUE?                        │
├─────────────────────────────────────────────────────────────────┤
│  USA LISTA cuando:          │  USA GENERADOR cuando:           │
│  • Necesitas indexar [i]    │  • Procesas una vez (pipeline)   │
│  • Necesitas len()          │  • Datos no caben en memoria     │
│  • Iteras múltiples veces   │  • Es infinito o muy largo       │
│  • Necesitas .append()      │  • Cadena de transformaciones    │
│  • Debugueas (ver contenido) │  • Producción bajo demanda      │
└─────────────────────────────────────────────────────────────────┘
""")

---

## 7. itertools con Generadores

El módulo `itertools` amplía el poder de los generadores con herramientas de composición.

In [ ]:
from itertools import cycle, chain, islice, tee, count

# cycle: repite un iterable infinitamente
turnos = cycle(["Alice", "Bob", "Carol"])
print("Primeros 7 turnos:")
for i, nombre in enumerate(islice(turnos, 7)):
    print(f"  Turno {i + 1}: {nombre}")

In [ ]:
# chain: concatenar generadores sin crear listas intermedias
def rango_par(inicio, fin):
    return (x for x in range(inicio, fin) if x % 2 == 0)

def rango_impar(inicio, fin):
    return (x for x in range(inicio, fin) if x % 2 != 0)

todos = chain(rango_par(0, 10), rango_impar(0, 10))
print(f"Pares + Impares (0-9): {list(todos)}")

# chain.from_iterable: aplanar listas de listas
listas = [[1, 2], [3, 4], [5, 6]]
aplanado = list(chain.from_iterable(listas))
print(f"Aplanado: {aplanado}")

In [ ]:
# count: generador infinito de enteros
print("Primeros 5 con count(10, 2):")
for v in islice(count(10, 2), 5):
    print(f"  {v}", end="")
print()

In [ ]:
# tee: duplicar un generador (¡con precaución!)
def gen_lento():
    """Simula generador con costo."""
    for i in range(5):
        print(f"  [produciendo {i}]")
        yield i

# ¡CUIDADO! tee almacena en memoria todo lo que no se ha consumido
print("Usando tee para duplicar:")
original, copia = tee(gen_lento())

print("\nConsumiendo original:")
print(f"  Original: {list(original)}")

print("\nConsumiendo copia:")
print(f"  Copia: {list(copia)}")

print("\n⚠ tee es útil cuando ambos consumidores avanzan similarmente.")
print("  Si uno avanza mucho más, tee acumula memoria.")

---

## Diagrama: Evaluación Perezosa vs Eager

```
EAGER (List Comprehension)              LAZY (Generator Expression)
═════════════════════════              ═══════════════════════════

 datos ──▶ [f(x) for x] ──▶ Lista     datos ──▶ (f(x) for x) ──▶ Generador
                 │                                   │
                 ▼                                   ▼
        ┌─────────────────┐                 ┌─────────────────┐
        │ f(x₀) calculado │                 │ Sólo la fórmula │
        │ f(x₁) calculado │                 │ en memoria      │
        │ f(x₂) calculado │                 └────────┬────────┘
        │ ...              │                          │
        │ f(xₙ) calculado │                          │ next()
        └────────┬────────┘                          ▼
                 │                          ┌─────────────────┐
                 ▼                          │ Calcula SOLO    │
        [v₀, v₁, v₂, ..., vₙ]             │ next valor      │
        Todo en RAM al instante            └────────┬────────┘
                                                   │ next()
                                                   ▼
                                          ┌─────────────────┐
                                          │ Calcula SOLO    │
                                          │ next valor      │
                                          └─────────────────┘
                                                   ...
                                          Memoria: O(1) constante

Eager: O(n) memoria          Lazy: O(1) memoria
Eager: acceso indexado        Lazy: solo secuencial
Eager: reutilizable           Lazy: agotado tras uso
```

## Tabla Comparativa: Lista vs Generador

| Característica | List Comprehension `[]` | Generator Expression `()` |
|---|---|---|
| **Memoria** | O(n) - almacena todos | O(1) - solo fórmula + estado |
| **Creación** | Lenta para n grande | Instantánea |
| **Acceso por índice** | Sí (`lista[i]`) | No |
| **`len()`** | Sí (`len(lista)`) | No |
| **Reutilización** | Sí (iteras múltiples) | No (se agota, `StopIteration`) |
| **`sys.getsizeof`** | Variable (crece con n) | Constante (~120-200 bytes) |
| **Uso ideal** | Datos pequeños, acceso directo | Pipelines, datos masivos, infinitos |
| **Ejemplo** | `[x**2 for x in range(n)]` | `(x**2 for x in range(n))` |

---

## Ejercicios

### Ejercicio 1 (Guiado): Comprehension con walrus

Dada una lista de precios con posibles valores `None`, extrae solo los válidos y calcula el IVA (21%), mostrando el precio con y sin IVA en un diccionario.

In [ ]:
precios_raw = [25.5, None, 19.99, None, 45.0, 12.75, None, 99.95, 8.5]

# Completa: usa walrus para calcular iva una sola vez
# Resultado esperado: dict con precio_base y precio_con_iva (solo válidos)
catalogo = {
    f"item_{i}": {
        "base": round(p, 2),
        "con_iva": round(p * 1.21, 2)
    }
    for i, p in enumerate(precios_raw) if p is not None
}

for k, v in catalogo.items():
    print(f"{k}: base=${v['base']}, con IVA=${v['con_iva']}")

### Ejercicio 2 (Guiado): Generador de secuencias de Fibonacci con límite

Crea un generador que produzca Fibonacci pero que se detenga cuando el siguiente valor **exceda** un máximo dado.

In [ ]:
def fibonacci_hasta(maximo):
    """Genera Fibonacci hasta que el siguiente valor exceda maximo."""
    a, b = 0, 1
    while a <= maximo:
        yield a
        a, b = b, a + b

# Probar con máximo 100
print(f"Fibonacci hasta 100: {list(fibonacci_hasta(100))}")
print(f"Fibonacci hasta 1000: {list(fibonacci_hasta(1000))}")

### Ejercicio 3 (Guiado): Pipeline de procesamiento con itertools

Construye un pipeline que:
1. Genere números del 1 al 50
2. Filtre solo los primos (usa tu criba)
3. Tome los primeros 5
4. Multiplique cada uno por su posición (1-indexed)

In [ ]:
from itertools import islice, count

def es_primo(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

# Pipeline funcional
numeros = (n for n in range(1, 51))
primos = (n for n in numeros if es_primo(n))
primeros_5 = islice(primos, 5)
con_posicion = ((i, p, i * p) for i, p in enumerate(primeros_5, 1))

print("Pipeline de primos:")
for pos, primo, producto in con_posicion:
    print(f"  Posición {pos}: primo={primo}, producto={producto}")

### Ejercicio 4 (Independiente): Generador Paginador de Datos

Crea una función `paginar(datos, tamano_pagina)` que:

1. Reciba una lista de elementos (puede ser muy grande)
2. Devuelva un generador que produce **tuplas** de elementos por página
3. La última página puede tener menos elementos
4. Cada página sea una tupla (inmutable) para diferenciarse de una lista

**Bonus**: Agrega un parámetro `skip` para saltar las primeras N páginas.

In [ ]:
def paginar(datos, tamano_pagina, skip=0):
    """
    Generador que produce páginas de datos.
    
    Args:
        datos: iterable de elementos
        tamano_pagina: elementos por página
        skip: páginas iniciales a saltar
    
    Yields:
        Tupla con los elementos de cada página
    """
    iterador = iter(datos)
    pagina_actual = 0
    
    while True:
        elementos = []
        for _ in range(tamano_pagina):
            try:
                elementos.append(next(iterador))
            except StopIteration:
                break
        
        if not elementos:
            break
        
        pagina_actual += 1
        if pagina_actual > skip:
            yield tuple(elementos)

# Probar con datos ficticios
datos_prueba = list(range(1, 26))  # 25 elementos
print("Todas las páginas (tamaño 5):")
for i, pagina in enumerate(paginar(datos_prueba, 5)):
    print(f"  Página {i + 1}: {pagina}")

print("\nSaltando las primeras 2 páginas:")
for i, pagina in enumerate(paginar(datos_prueba, 5, skip=2)):
    print(f"  Página {i + 3}: {pagina}")

In [ ]:
# Verificación de que funciona con datos infinitos (generador como entrada)
def infinito():
    n = 1
    while True:
        yield n
        n += 1

print("Paginando un generador infinito (primeras 3 páginas):")
for i, pagina in enumerate(paginar(infinito(), 4)):
    print(f"  Página {i + 1}: {pagina}")
    if i >= 2:
        break

---

## Resumen

| Concepto | Sintáxis | Uso principal |
|---|---|---|
| **List comprehension** | `[expr for x in seq]` | Crear lista pequeña/moderada |
| **Dict comprehension** | `{k: v for x in seq}` | Mapear datos a diccionarios |
| **Set comprehension** | `{expr for x in seq}` | Eliminar duplicados |
| **Generador expression** | `(expr for x in seq)` | Pipeline de datos, ahorrar memoria |
| **Función generadora** | `def f(): yield x` | Estado complejo, lógica procedural |
| **`yield from`** | `yield from gen` | Delegar a sub-generadores |
| **`itertools`** | `islice, chain, cycle...` | Componer generadores eficientemente |

**Regla de oro**: Si los datos caben en memoria y necesitas reutilizar → **lista**. Si son masivos, infinitos o de un solo uso → **generador**.